# geoarrow_pyarrow

## Example 1: one geometry column in three encodings

In [ ]:
# Example 1: ga.array reads WKT; as_wkb and as_geoarrow convert it, and as_wkt brings it back
import geoarrow.pyarrow as ga

wkt = ga.array(["POINT (0 1)", "POINT (2 3)"])
wkb = ga.as_wkb(wkt)
native = ga.as_geoarrow(wkt)
assert wkt.type.extension_name == "geoarrow.wkt" and wkb.type.extension_name == "geoarrow.wkb"
assert native.type.extension_name == "geoarrow.point"
assert ga.as_wkt(native).storage.to_pylist() == ["POINT (0 1)", "POINT (2 3)"]
print("Example 1:", wkt.type, "->", wkb.type, "->", native.type)

## Example 2: coordinates out of a point array

In [ ]:
# Example 2: point_coords returns the x and y arrays
import numpy as np
import geoarrow.pyarrow as ga

x, y = ga.point_coords(ga.as_geoarrow(["POINT (0 1)", "POINT (1 2)", "POINT (5 -3)"]))
assert np.array_equal(np.asarray(x), [0.0, 1.0, 5.0]) and np.array_equal(np.asarray(y), [1.0, 2.0, -3.0])
print("Example 2: x", np.asarray(x).tolist(), "y", np.asarray(y).tolist())

## Example 3: per-row boxes and the overall box

In [ ]:
# Example 3: box gives one bounding box per geometry, box_agg one for the whole column
import geoarrow.pyarrow as ga

geoms = ga.array(["POINT (0 1)", "LINESTRING (2 3, 5 -1)"])
per_row = ga.box(geoms).storage.to_pylist()
overall = ga.box_agg(geoms).as_py()
assert per_row[1] == {"xmin": 2.0, "ymin": -1.0, "xmax": 5.0, "ymax": 3.0}, per_row
assert overall == {"xmin": 0.0, "ymin": -1.0, "xmax": 5.0, "ymax": 3.0}, overall
print("Example 3:", per_row, "|", overall)

## Example 4: points from coordinate arrays, with a CRS

In [ ]:
# Example 4: make_point builds a native point array and carries the CRS in the type
import geoarrow.pyarrow as ga

points = ga.make_point([10.0, 20.0], [1.0, 2.0], crs=ga.OGC_CRS84)
assert points.type.extension_name == "geoarrow.point" and points.type.crs == ga.OGC_CRS84
assert ga.format_wkt(points).to_pylist() == ["POINT (10 1)", "POINT (20 2)"]
print("Example 4:", points.type.crs, ga.format_wkt(points).to_pylist())

## Example 5: hand the column to GeoPandas

In [ ]:
# Example 5: to_geopandas turns a GeoArrow array into a GeoSeries of shapely geometries
import geoarrow.pyarrow as ga

series = ga.to_geopandas(ga.array(["POINT (0 1)", "POLYGON ((0 0, 2 0, 2 2, 0 2, 0 0))"]))
assert series.geom_type.tolist() == ["Point", "Polygon"] and series.area.tolist() == [0.0, 4.0]
print("Example 5:", type(series).__name__, series.geom_type.tolist(), series.area.tolist())